In [ ]:
import geopandas as gpd
import pandas as pd

# sf stands for shape file
sf = gpd.read_file("../data/landing/sa2/SA2_2021_AUST_GDA2020.shp")
income_data = pd.read_csv("../data/curated/suburb_info.csv")

sf.head()

In [ ]:
sf = sf.loc[sf["STE_NAME21"] == "Victoria"]

In [ ]:
sf['geometry'] = sf['geometry'].to_crs("+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs")

In [ ]:
income_data.head()

In [ ]:
print(type(income_data['SA2_codes'].iloc[1]))

print(type(sf['SA2_CODE21'].iloc[1]))

In [ ]:
sf['SA2_CODE21'] = sf['SA2_CODE21'].astype(int)

In [ ]:
gdf = gpd.GeoDataFrame(
    income_data.merge(sf, right_on='SA2_CODE21', left_on='SA2_codes', how='left')
)

gdf.head()

In [ ]:
# create a JSON 
geoJSON = gdf[['SA2_codes', 'geometry']].drop_duplicates('SA2_codes').to_json()

# print the first 300 chars of the json
print(geoJSON[:300])

In [ ]:
import folium

In [ ]:
_map = folium.Map(location=[-37.020100, 144.964600], tiles="Stamen Terrain", zoom_start=7)

# refer to the folium documentations on how to plot aggregated data.
_map.add_child(folium.Choropleth(
    geo_data=geoJSON,
    name='choropleth',
))

_map

In [ ]:
gdf.columns

In [ ]:
df = gdf[["SA2_codes", 'Median equivalised total household income (weekly) ($)']]

#df = df.rename(columns={"LGA_CODE23": "LocationID"})
df.head()

In [ ]:
m = folium.Map(location=[-37.020100, 144.964600], tiles="Stamen Terrain", zoom_start=7)

#geoJSON = gdf[['LGA_CODE23', 'geometry']].drop_duplicates('LGA_CODE23').to_json()

# refer to the folium documentations on more information on how to plot aggregated data.
c = folium.Choropleth(
    geo_data=geoJSON, # geoJSON 
    name='choropleth', # name of plot
    data=df.reset_index(), # data source
    columns=['SA2_codes','Median equivalised total household income (weekly) ($)'], # the columns required
    key_on= 'properties.SA2_codes', # this is from the geoJSON's properties
    fill_color='YlOrRd', # color scheme
    nan_fill_color='black',
    legend_name='Proportion of Trip Earnings USD$'
)

c.add_to(m)

m